In [7]:
import os, sys

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from typing import Literal
from pydantic import BaseModel, Field

In [8]:
project_root = os.path.dirname(os.getcwd())
sys.path.insert(0, project_root)

from utility.env_util import get_api_key

find_api = "OPENAI_API_KEY"
api_key = get_api_key(find_api)

In [9]:
# ① 모델 생성
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,  # 말이 길어지는 걸 방지
    max_completion_tokens=100,    # 응답 최대 길이 제한, 최신 버전에는 max_tokens 대신 사용 
)

In [11]:
# ② 메시지 기반 호출 (카페 역할극)
messages = [
    SystemMessage(content="너는 카페에서 일하는 직원이다. 손님에게 짧고 친절하게 응대해줘."),
    HumanMessage(content="안녕하세요. 케이크랑 음료 추천해 주세요."),
]
print('# StrOutputParser를 적용하기 이전입니다.')
model.invoke(messages)
'''
AIMessage(content='안녕하세요! 저희 인기 케이크는 초코 케이크와 레드벨벳 케이크입니다. 음료는 아메리카노와 바닐라 라떼가 잘 어울려요. 어떤 걸 드시고 싶으신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 48, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D29rFHwYQuCouLB5IsUHRCrnQGOGo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bf8de-f5b5-7fb2-95fb-b70ef5f13aba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 60, 'total_tokens': 108, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})
'''

# StrOutputParser를 적용하기 이전입니다.


"\nAIMessage(content='안녕하세요! 저희 인기 케이크는 초코 케이크와 레드벨벳 케이크입니다. 음료는 아메리카노와 바닐라 라떼가 잘 어울려요. 어떤 걸 드시고 싶으신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 48, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D29rFHwYQuCouLB5IsUHRCrnQGOGo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bf8de-f5b5-7fb2-95fb-b70ef5f13aba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 60, 'total_tokens': 108, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})\n"

In [12]:
# ③ OutputParser + LCEL 체인
parser = StrOutputParser()

# 인보크한 모델을 출력 parser에게 다시 인보크시켜서 필요한 문자열만 출력해 줍니다.
result = model.invoke(messages)
print('# StrOutputParser를 적용시킨 결과입니다.')
parser.invoke(result)
'''
'안녕하세요! 저희 카페의 초코 케이크와 아메리카노 조합이 정말 인기 있어요. 달콤한 케이크와 진한 커피가 잘 어울린답니다. 혹시 다른 취향이 있으신가요?'
'''

# StrOutputParser를 적용시킨 결과입니다.


"\n'안녕하세요! 저희 카페의 초코 케이크와 아메리카노 조합이 정말 인기 있어요. 달콤한 케이크와 진한 커피가 잘 어울린답니다. 혹시 다른 취향이 있으신가요?'\n"

In [13]:
# 파이프(|) 연산자는 Runnable 합성 연산자라고 하며, 왼쪽의 출력 정보를 오른쪽의 입력 정보로 입력하는 역할을 합니다.
chain = model | parser
chain.invoke(messages)
'''
'안녕하세요! 저희 인기 케이크는 초코 케이크와 치즈 케이크입니다. 음료는 아메리카노나 바닐라 라떼가 잘 어울려요. 어떤 걸 드시겠어요?'
'''

"\n'안녕하세요! 저희 인기 케이크는 초코 케이크와 치즈 케이크입니다. 음료는 아메리카노나 바닐라 라떼가 잘 어울려요. 어떤 걸 드시겠어요?'\n"

In [14]:
# ④ 프롬프트 템플릿
# ChatPromptTemplate로 카페 상황 일반화
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {place}에서 일하는 {role}이다. 손님에게 짧고 친절하게 대답해줘."
human_template = "{menu} 추천해 주세요."

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", human_template),
])

In [15]:
# ⑤ LCEL 체인 구성
chain = prompt_template | model | parser

chain.invoke({
    "place": "카페",
    "role": "직원",
    "menu": "케이크랑 음료",
})
'''
'저희 인기 메뉴인 초코 케이크와 카라멜 라떼를 추천해 드려요! 정말 맛있답니다.'
'''

"\n'저희 인기 메뉴인 초코 케이크와 카라멜 라떼를 추천해 드려요! 정말 맛있답니다.'\n"

In [16]:
print('# 여러 개의 입력은 batch 처리(한 번에 처리)를 사용하면 좋습니다.')
inputs = [
    {"place": "카페", "role": "직원", "menu": "음료"},
    {"place": "카페", "role": "직원", "menu": "빵"},
]

results = chain.batch(inputs)

for input_data, result in zip(inputs, results):
    print(f"# 입력값 : {input_data}")
    print(f"# 결과   : {result}")
    print("-" * 30)

# results = chain.batch(inputs)
#
# for rst in results:
#     print(rst)
#     print('-'*30)
# # end for
'''
어떤 종류의 음료를 원하시나요? 커피, 차, 또는 스무디 중에서 선택해 드릴 수 있어요!
------------------------------
저희의 크루아상과 바게트가 특히 인기 있어요! 둘 다 맛있으니 한번 드셔보세요.
------------------------------
'''

# 여러 개의 입력은 batch 처리(한 번에 처리)를 사용하면 좋습니다.
# 입력값 : {'place': '카페', 'role': '직원', 'menu': '음료'}
# 결과   : 어떤 종류의 음료를 원하시는지 말씀해 주시면 추천해 드릴게요! 커피, 차, 또는 스무디 중에서요?
------------------------------
# 입력값 : {'place': '카페', 'role': '직원', 'menu': '빵'}
# 결과   : 저희의 크루아상과 바게트가 정말 인기 있어요! 둘 다 맛있으니 한번 드셔보세요.
------------------------------


'\n어떤 종류의 음료를 원하시나요? 커피, 차, 또는 스무디 중에서 선택해 드릴 수 있어요!\n------------------------------\n저희의 크루아상과 바게트가 특히 인기 있어요! 둘 다 맛있으니 한번 드셔보세요.\n------------------------------\n'

In [17]:
# 반드시 { answer: 문자열, emotion: '친절' 또는 '중립' }의 구조로 대답해 줘야되.
class CafeResponse(BaseModel):
    answer: str = Field(description="카페 직원의 응답 (짧게)")
    emotion: Literal["친절", "중립"] = Field(description="응답의 분위기")

# "AI야, 너의 답변은 반드시 CafeResponse 구조로 만들어야 되."
# with_structured_output() : 출력 형식은 반드시 CafeResponse 스키마를 따라서 처리해야 합니다.(출력 형식 강제 지정)
parser_condition = model.with_structured_output(CafeResponse)

new_chain = prompt_template | parser_condition

new_chain.invoke({
    "place": "카페",
    "role": "직원",
    "menu": "초코 케이크랑 커피",
})
'''
CafeResponse(answer='초코 케이크와 잘 어울리는 커피는 아메리카노나 카푸치노를 추천드려요! 달콤한 초코와 쌉싸름한 커피의 조화가 정말 맛있답니다.', emotion='중립')
'''

c:\Python311\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CafeResponse(answer='초...요.', emotion='중립'), input_type=CafeResponse])
  return self.__pydantic_serializer__.to_python(


"\nCafeResponse(answer='초코 케이크와 잘 어울리는 커피는 아메리카노나 카푸치노를 추천드려요! 달콤한 초코와 쌉싸름한 커피의 조화가 정말 맛있답니다.', emotion='중립')\n"